# Lab 5a: Community Detection (Modularity · Louvain · Spectral)
**Course:** IS353 - Social Network Analysis, UIT, VNUHCM  
**Lecturer:** Dr. Tran Hung Nghiep, 2025  

---

**Goals:**
- Understand and compute **modularity** \(Q\) for a partition.
- Apply **Louvain**, **Spectral Clustering**, and other methods like **Label Propagation**,  **Greedy Modularity**, **Girvan-Newman**.
- Compare results: number of communities, modularity, and runtime.
- Visualize partitions and export to Gephi for further inspection.


## 0. Setup

In [ ]:

# Setup
import time
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from collections import defaultdict
from itertools import islice
from networkx.algorithms.community.quality import modularity as nx_modularity

def draw_partition(G, partition, pos=None, title="Communities"):
    if pos is None:
        pos = nx.spring_layout(G, seed=42)
    if isinstance(partition, dict):
        node2com = partition
    else:
        node2com = {}
        for i, com in enumerate(partition):
            for v in com:
                node2com[v] = i
    colors = [node2com[n] for n in G.nodes()]
    plt.figure(figsize=(8,6))
    nx.draw(G, pos, with_labels=True, node_color=colors, node_size=400, font_size=9)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()
    return pos

def partition_as_sets_from_dict(partition_dict):
    d = {}
    for n, c in partition_dict.items():
        d.setdefault(c, set()).add(n)
    return list(d.values())

def partition_stats(G, partition_sets):
    sizes = sorted([len(c) for c in partition_sets], reverse=True)
    return {"k": len(partition_sets), "sizes": sizes, "modularity": nx_modularity(G, partition_sets)}

def print_stats(label, stats, elapsed=None):
    line = f"{label}: k={stats['k']}, sizes={stats['sizes']}, Q={stats['modularity']:.4f}"
    if elapsed is not None:
        line += f", time={elapsed:.3f}s"
    print(line)



## 1) Load Graph


In [ ]:

G = nx.karate_club_graph()
print(G)
pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(8,6))
nx.draw(G, pos, with_labels=True, node_color="lightgray", node_size=400, font_size=9)
plt.title("Karate Club — Base Graph")
plt.axis("off")
plt.tight_layout()
plt.show()



## 2) Modularity Q (Baseline with Label Propagation)


In [ ]:

from networkx.algorithms.community import label_propagation_communities

t0 = time.time()
lp_partition = list(label_propagation_communities(G))
t1 = time.time()
lp_stats = partition_stats(G, lp_partition)
print_stats("Label Propagation", lp_stats, t1 - t0)
pos = draw_partition(G, lp_partition, pos=pos, title="Label Propagation (baseline)")



## 3) Louvain


In [ ]:

import community.community_louvain as community_louvain

t0 = time.time()
louvain_partition_dict = community_louvain.best_partition(G)
t1 = time.time()
louvain_sets = partition_as_sets_from_dict(louvain_partition_dict)
louvain_stats = partition_stats(G, louvain_sets)
print_stats("Louvain", louvain_stats, t1 - t0)
pos = draw_partition(G, louvain_partition_dict, pos=pos, title="Louvain communities")



## 4) Spectral Clustering (Fiedler + Sweep Cut)


In [ ]:

import numpy.linalg as LA

L = nx.normalized_laplacian_matrix(G).astype(float).todense()
eigvals, eigvecs = LA.eigh(L)
idx_sorted = np.argsort(eigvals)
fiedler = np.asarray(eigvecs[:, idx_sorted[1]]).flatten()

order = np.argsort(fiedler)
best_Q = -1.0
best_split = None
nodes = list(G.nodes())

for cut_pos in range(1, len(nodes)):
    left_idx = order[:cut_pos]
    right_idx = order[cut_pos:]
    left = set(nodes[i] for i in left_idx)
    right = set(nodes[i] for i in right_idx)
    comms = [left, right]
    Q = nx_modularity(G, comms)
    if Q > best_Q:
        best_Q = Q
        best_split = (left, right)

spectral_partition = [best_split[0], best_split[1]]
spectral_stats = partition_stats(G, spectral_partition)
print_stats("Spectral (2-way, best sweep)", spectral_stats)
pos = draw_partition(G, spectral_partition, pos=pos, title="Spectral clustering (best sweep cut)")



## 5) Greedy Modularity & Girvan-Newman


In [ ]:

from networkx.algorithms.community import greedy_modularity_communities, girvan_newman

t0 = time.time()
gmc_partition = list(greedy_modularity_communities(G))
t1 = time.time()
gmc_stats = partition_stats(G, gmc_partition)
print_stats("Greedy Modularity (CNM)", gmc_stats, t1 - t0)
pos = draw_partition(G, gmc_partition, pos=pos, title="Greedy Modularity (CNM)")

gn = girvan_newman(G)
gn_levels = list(islice(gn, 2))
for li, comms in enumerate(gn_levels, start=1):
    comms = [set(c) for c in comms]
    stats = partition_stats(G, comms)
    print_stats(f"Girvan-Newman level {li}", stats)
    pos = draw_partition(G, comms, pos=pos, title=f"Girvan-Newman level {li}")



## 6) Summary Table
**Requirements:** log running time for each algorithm.

In [ ]:

import pandas as pd

rows = []
rows.append(["Label Propagation", lp_stats["k"], lp_stats["sizes"], lp_stats["modularity"], None])
rows.append(["Spectral (Fiedler)", spectral_stats["k"], spectral_stats["sizes"], spectral_stats["modularity"], None])
rows.append(["Greedy Modularity (CNM)", gmc_stats["k"], gmc_stats["sizes"], gmc_stats["modularity"], None])

if 'louvain_stats' in globals():
    rows.append(["Louvain", louvain_stats["k"], louvain_stats["sizes"], louvain_stats["modularity"], None])

for li, comms in enumerate(gn_levels, start=1):
    stats = partition_stats(G, [set(c) for c in comms])
    rows.append([f"Girvan–Newman level {li}", stats["k"], stats["sizes"], stats["modularity"], None])

df = pd.DataFrame(rows, columns=["Method", "k", "sizes", "Q (modularity)", "time (s)"])
df



## 7) Export to Gephi


In [ ]:

node2com = {}
for i, com in enumerate(gmc_partition):
    for v in com:
        node2com[v] = i
nx.set_node_attributes(G, node2com, "community")
nx.write_gexf(G, "karate_communities.gexf")
print("Wrote karate_communities.gexf (node attribute: community)")



## Questions

1. **Compare results** across all methods (Louvain, Spectral Clustering, Greedy Modularity, Girvan–Newman (level 1–3), and Label Propagation) by number of components, size, modularity, running time.
2. **Compare visually** across all methods. What structural differences do you observe?
3. Which method best matches the intuitive split of the Karate Club? Why?
4. (Extra) Re-run on the Facebook dataset, which method scaled best, and what patterns did you see?


# Appendix: Compute modularity Q using the general formula

In [ ]:
import networkx as nx
import numpy as np

def modularity_ordered_pair(G, communities):
    """
    Compute modularity Q(G, S) using the ordered-pair sum formula:

        Q = 1/(2m) * sum_s sum_{i,j in s} [ A_ij - (k_i * k_j)/(2m) ]

    where:
        - A_ij = adjacency matrix (1 if edge exists)
        - k_i  = degree of node i
        - m    = number of edges (undirected)
    """
    # adjacency and degree
    nodes = list(G.nodes())
    A = nx.to_numpy_array(G, nodelist=nodes)
    k = np.array([G.degree(n) for n in nodes])
    m = G.number_of_edges()
    node_index = {n: i for i, n in enumerate(nodes)}

    Q = 0.0
    for com in communities:
        idx = [node_index[n] for n in com]
        A_sub = A[np.ix_(idx, idx)]
        k_sub = k[idx]
        term = A_sub.sum() - (k_sub.sum()**2) / (2*m)
        Q += term
    return Q / (2*m)


In [ ]:
G = nx.Graph()
G.add_edges_from([
    ("A","B"),("B","C"),("C","A"),      # triangle 1
    ("D","E"),("E","F"),("F","D"),      # triangle 2
    ("B","D"),("C","E")                 # cross edges
])

S = [ {"A","B","C"}, {"D","E","F"} ]

Q = modularity_ordered_pair(G, S)
print("Modularity Q =", Q)


Modularity Q = 0.25
